In [14]:
# Polymarket Test
from src.handlers.polymarket import PolymarketHandler
from src.utils.utils import *
from dotenv import load_dotenv
import psycopg2
from psycopg2 import OperationalError
import copy
import requests
from datetime import datetime
from huggingface_hub import list_repo_files, hf_hub_download, login
import pprint
import time
import duckdb

import os

# Load environment variables
load_dotenv()

# Make a PolymarketHandler instance
polymarketHandler = PolymarketHandler(
    polymarketAPIkey = os.getenv("polymarketAPI_key"),
    web3APIkey = os.getenv("alchemyAPI_key"),
    provider = "alchemy"
)


In [60]:

result = getFromSubgraph(
    "a03be5e9f766c0de3ca0441519f1d9ca",
    "7fu2DWYK93ePfzB24c2wrP94S3x4LGHUrQxphhoEypyY",
    """
    {
        orderFilledEvents(block: {number: 84997444}) {
            id
            transactionHash
            timestamp
            orderHash
            blockNumber
            takerAssetId
            makerAssetId
        }
    }
    """
)
txHashes_ql = [x["transactionHash"] for x in result["data"]["orderFilledEvents"]]

with open("src/ABIs/polygon/polymarket_exchange_CTF_v1.abi", "r") as f:
    abi = json.load(f)
contract_v1_CTF = polymarketHandler.w3["polygon"].eth.contract(address="0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E", abi=abi)

result_2 = polymarketHandler.w3["polygon"].eth.get_logs({
    "address": "0xC5d563A36AE78145C45a50134d48A1215220f80a",
    "fromBlock": 84997444,
    "toBlock": 84997444,
    "topics": ["0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6"]
})
txHashes_rpc = [x.transactionHash.hex() for x in result_2]

In [61]:
df_rpc = pd.DataFrame(result_2)
df_rpc.drop(["data", "topics"] ,axis = 1, inplace=True)
df_rpc['transactionHash'] = df_rpc['transactionHash'].apply(lambda x: f"0x{x.hex()}" if hasattr(x, 'hex') else x)
len(df_rpc.transactionHash.unique())

36

In [62]:
df_ql = pd.DataFrame(result["data"]["orderFilledEvents"])
df_ql.drop(["id"] ,axis = 1, inplace=True)

len(df_ql.transactionHash.unique())

42

In [63]:
print(df_ql.sort_values(by='blockNumber').iloc[-1].transactionHash)
df_ql.sort_values(by='blockNumber')

0x000000573a7793bc268afbf51cdf0c26a8df45c64a09aeaa852c3aed7400da85


,transactionHash,timestamp,orderHash,blockNumber,takerAssetId,makerAssetId
34,0x000000eae9b9fbc15b35e567f3ee3390d75400a0930b...,1748971146,0xa81b8978de0463527a2c1e3f0c64ff3a9864a1719b6f...,72320948,9022242446965460992675148513465279956952237358...,0
35,0x000000eae9b9fbc15b35e567f3ee3390d75400a0930b...,1748971146,0xe9a2c4afaa6b9e31a4a17aed91f5ad27ad5cfa4d114e...,72320948,0,9022242446965460992675148513465279956952237358...
38,0x000000f9fea26b5ea994ff8a5413742bfc85ea7471f2...,1752993663,0x5af33a58f9e7fc47c25710d98c06920ef0c5293877ed...,74179856,5265040868823602339054725031981699427002830201...,0
39,0x000000f9fea26b5ea994ff8a5413742bfc85ea7471f2...,1752993663,0x89050d0f06dc8c1cd55410c2438f46172ef60d8e2a20...,74179856,2599244875573717645039155436674359184743198118...,0
10,0x00000035bd23406307532e86f280c29422bd0f69e868...,1753832393,0x1d3c0030b6c1004a7314b5e651d7364560cb3702b9cd...,74573090,7150931994402382761660133607857978344532686546...,0
...,...,...,...,...,...,...
98,0x000002153883929788ac56a14d635a5c3b31cec3661d...,1774825916,0x2bb72b6b059fd22febdaf81cd7275e0e122d0c38dd0c...,84852940,3152036997913854476001090222302500284337810054...,0
71,0x0000016f701bb27d6a3aed2a576f8fb9dbef09c51c9f...,1774955823,0x7617037dd776d8a7ee96e1d18c856cafc3fcd108049c...,84917893,8388098268388988596513372750010009535114539620...,0
72,0x0000016f701bb27d6a3aed2a576f8fb9dbef09c51c9f...,1774955823,0xf3271a14f43550fc96caea3e347b8068f6a90434c0da...,84917893,3188346274911955638808679939991975363723255521...,0
16,0x000000573a7793bc268afbf51cdf0c26a8df45c64a09...,1775114923,0x2a42445bb75b60a674f4256b406823ba9ce060bd7d01...,84997443,0,1882383899744387865687995259050252452655650403...


In [64]:
for tx in df_rpc.transactionHash.unique():
    if tx not in df_ql.transactionHash:
        # print(f"Transaction {tx} is in RPC but not in QL")
        pass
    else:
        print(f"Transaction {tx} is in both RPC and QL")
        pass

In [2]:
result_2 = polymarketHandler.w3["polygon"].eth.get_logs({
    "address": "0xe2222d279d744050d28e00520010520000310F59",
    "fromBlock": 88997444,
    "toBlock": 88997444,
    "topics": ["0xd543adfd945773f1a62f74f0ee55a5e3b9b1a28262980ba90b1a89f2ea84d8ee"]
})
len(result_2)

18

In [3]:
result_2[0]

AttributeDict({'address': '0xe2222d279d744050d28e00520010520000310F59',
 'topics': [HexBytes('0xd543adfd945773f1a62f74f0ee55a5e3b9b1a28262980ba90b1a89f2ea84d8ee'),
  HexBytes('0x6bfb35cb0f4544ff5e152fafd5d68512df46684134a7bf35e6e9f911946031cd'),
  HexBytes('0x00000000000000000000000021064fd320bfd5a86f8c92a94d3209edf4154dea'),
  HexBytes('0x0000000000000000000000007be83219c0d406572e07385dd12e1fbeb1bb0f50')],
 'data': HexBytes('0x0000000000000000000000000000000000000000000000000000000000000000beda89649b2a6ea0122c06c69c56a90b7ddedb89b4c1cc5793e6559789af7dbe0000000000000000000000000000000000000000000000000000000000107ac000000000000000000000000000000000000000000000000000000000002dc6c0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000'),
 'blockNumber': 88997444,
 'transactionHash': HexBytes('0x22ae7b382a7f1f9222591bc150188738d9a79f4b4e457326357d5a8613

In [4]:
polymarketHandler._NEGRISK_V2_OrderFilled_fastDecode(result_2[0])

{'tokenId': '86325562142527339371528565685614438265267177138486196674163028053281653882302',
 'side': 0,
 'makerAmountFilled': 1080000,
 'takerAmountFilled': 3000000,
 'orderHash': '0x6bfb35cb0f4544ff5e152fafd5d68512df46684134a7bf35e6e9f911946031cd',
 'taker': '0x7Be83219C0d406572E07385Dd12e1fBEB1bB0F50',
 'maker': '0x21064fD320bFd5A86f8c92a94d3209edf4154deA',
 'fee': 0,
 'platform': 'polymarket',
 'platformVersion': 'negrisk_v2'}

In [20]:
f"0x{result_2[0]["transactionHash"].hex()}"

'0x22ae7b382a7f1f9222591bc150188738d9a79f4b4e457326357d5a8613b7f32b'

In [4]:
df = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT * FROM data "))
df.columns

Index(['block_number', 'block_timestamp', 'tx_hash', 'platform_name',
       'platform_version', 'taker', 'maker', 'token_id', 'slug',
       'condition_id', 'price', 'taker_side', 'amount', 'amount_asset', 'fee',
       'fee_asset', 'extra_data'],
      dtype='str')

In [22]:
blockNumber = getBlockNumberFromTS(polymarketHandler.w3["polygon"], 1777374040)
blockNumber

86126998

In [23]:
_max = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_017.parquet", "SELECT MAX(block_timestamp) AS largest_value FROM data")).iloc[0,0]
_min = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MIN(block_timestamp) AS largest_value FROM data")).iloc[0,0]
print(_min)
print(_max)
print("time difference with prev batch:", (_min-_max)/60/24, "days")

_max = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_017.parquet", "SELECT MAX(block_number) AS largest_value FROM data")).iloc[0,0]
_min = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MIN(block_number) AS largest_value FROM data")).iloc[0,0]
print(_min)
print(_max)
print("block difference with prev batch:", (_min-_max), "blocks")

_min = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MIN(block_timestamp) AS largest_value FROM data")).iloc[0,0]
_max = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MAX(block_timestamp) AS largest_value FROM data")).iloc[0,0]
print("time difference:", (_max-_min)/60/24, "days")

_min = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MIN(block_number) AS largest_value FROM data")).iloc[0,0]
_max = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT MAX(block_number) AS largest_value FROM data")).iloc[0,0]
print("block difference:", (_max-_min), "blocks")

1782380735
1777374040
time difference with prev batch: 3476.871527777778 days
89117160
-1
block difference with prev batch: 89117161 blocks
time difference: 13.332638888888889 days
block difference: 12799 blocks


In [14]:
df = pd.DataFrame(queryParquetFile("src/data/polymarketData/polymarket_trades_pt_018.parquet", "SELECT DISTINCT block_timestamp FROM data"))
df

,block_timestamp
0,1782399356
1,1782399361
2,1782399373
3,1782399391
4,1782399392
...,...
12787,1782381302
12788,1782381305
12789,1782381307
12790,1782381320
